# LG-PM major revision — reproducible re-analyses (~2 h on T4)

Everything except the raw images comes from the GitHub repo (code, phase artifacts, results)
and Hugging Face (8 large files, SHA-256 checked). Every script first **reproduces the
locked published numbers** (`[GATE OK]`) and stops with `[GATE FAIL]` otherwise.

**Settings:** Accelerator = GPU T4 x2 · Internet = On.

**Attach only the raw image datasets (Add Input):** TN3K_Data (contains DDTI) · TNUI-2021 ·
TN5000 · ThyUS2Path · DDTI: Thyroid Ultrasound Images (only if Phase 3e used it).
Do **not** attach the phase output datasets; the repo carries them.

In [ ]:
REPO_URL = "https://github.com/nihal4/lgpm-revision.git"   # <- edit after you push
import os, subprocess
if not os.path.exists("/kaggle/working/lgpm-revision"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/kaggle/working/lgpm-revision"], check=True)
os.chdir("/kaggle/working/lgpm-revision")
!git log -1 --oneline
!nvidia-smi --query-gpu=name,memory.total --format=csv
!ls /kaggle/input/datasets/*/ 2>/dev/null || ls /kaggle/input

## 0. Fetch the large artifacts from Hugging Face (checksummed, ~600 MB)

In [ ]:
!python scripts/fetch_large_files.py

## 1. DDTI label provenance (CPU, ~2 min) — R2.7

In [ ]:
!python scripts/s04_ddti_label_profile.py 2>&1 | tail -n 60

## 2. CPU re-analyses from saved predictions (~3 min) — R2.2, R2.3, R2.5

In [ ]:
!python scripts/s05_cpu_analyses.py 2>&1 | grep -E 'GATE|failed|Error|done'

## 3. ThyUS2Path operating points for all six models (GPU, ~15 min) — R2.2, R1.2

In [ ]:
!python scripts/s01_thyus2path_all_models.py 2>&1 | tail -n 80

## 4. Explanation controls on TN5000-test (GPU, ~10 min) — R2.4, R1.7

In [ ]:
!python scripts/s02_xai_controls.py 2>&1 | tail -n 60

## 5. Support-free decoder control (GPU, ~60–80 min) — R2.1, R1.3

In [ ]:
!python scripts/s03_support_free_decoder.py 2>&1 | tail -n 80

## 6. Package the outputs — download `revision_outputs.zip` from the Output panel and send it back

In [ ]:
!cd /kaggle/working && rm -f revision_outputs.zip && zip -qr revision_outputs.zip revision_outputs -x '*.pt'
!du -sh /kaggle/working/revision_outputs.zip
!grep -h 'GATE' /kaggle/working/revision_outputs/*/log.txt